# 02. Split GeoTIFF to 1024px Tiles

Split the merged V-World aerial GeoTIFF into fixed-size image tiles for polygon labeling.

**Input**
- `data/raw/vworld_aerial_test_area.tif`

**Output**
- `data/processed/tiles_1024/images/*.png` - 1024 x 1024 RGB tiles
- `data/processed/tiles_1024/tile_index.csv` - tile offsets and WGS84 bounds
- `data/processed/tiles_1024/tile_summary.json` - run summary
- `data/processed/tiles_1024/preview/contact_sheet.png` - quick visual check

Edge tiles are padded to 1024 x 1024. Use `valid_width` and `valid_height` in the index to identify the non-padded source area.

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import rasterio
from rasterio.transform import array_bounds
from rasterio.windows import Window

PROJECT_ROOT = Path("../..").resolve()
INPUT_PATH = PROJECT_ROOT / "data/raw/vworld_aerial_test_area.tif"

AREA_NAME = "vworld_aerial_test_area"
TILE_SIZE = 1024
OUTPUT_ROOT = PROJECT_ROOT / "data/processed/tiles_1024"
IMAGE_DIR = OUTPUT_ROOT / "images"
PREVIEW_DIR = OUTPUT_ROOT / "preview"
INDEX_PATH = OUTPUT_ROOT / "tile_index.csv"
SUMMARY_PATH = OUTPUT_ROOT / "tile_summary.json"
CONTACT_SHEET_PATH = PREVIEW_DIR / "contact_sheet.png"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing input GeoTIFF: {INPUT_PATH}")

print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_ROOT}")

In [ ]:
with rasterio.open(INPUT_PATH) as src:
    width, height = src.width, src.height
    crs = src.crs.to_string() if src.crs else None
    bounds = src.bounds
    dtype = src.dtypes[0]
    count = src.count

n_cols = math.ceil(width / TILE_SIZE)
n_rows = math.ceil(height / TILE_SIZE)
n_tiles = n_cols * n_rows

print(f"Source size: {width} x {height} px")
print(f"Bands: {count}, dtype: {dtype}, CRS: {crs}")
print(f"Bounds: {bounds}")
print(f"Tile size: {TILE_SIZE} x {TILE_SIZE} px")
print(f"Tile grid: {n_cols} x {n_rows} = {n_tiles} tiles")

In [ ]:
records = []

with rasterio.open(INPUT_PATH) as src:
    if src.count < 3:
        raise ValueError("Expected at least 3 bands for RGB output.")

    for row in range(n_rows):
        for col in range(n_cols):
            x_off = col * TILE_SIZE
            y_off = row * TILE_SIZE
            valid_width = min(TILE_SIZE, src.width - x_off)
            valid_height = min(TILE_SIZE, src.height - y_off)

            window = Window(x_off, y_off, valid_width, valid_height)
            data = src.read([1, 2, 3], window=window)
            data = np.moveaxis(data, 0, -1)

            tile = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.uint8)
            tile[:valid_height, :valid_width, :] = data.astype(np.uint8)

            tile_name = f"{AREA_NAME}_tile{TILE_SIZE}_r{row:02d}_c{col:02d}.png"
            tile_path = IMAGE_DIR / tile_name
            Image.fromarray(tile).save(tile_path)

            tile_transform = rasterio.windows.transform(window, src.transform)
            west, south, east, north = array_bounds(valid_height, valid_width, tile_transform)

            records.append({
                "tile_id": f"r{row:02d}_c{col:02d}",
                "file_name": tile_name,
                "row": row,
                "col": col,
                "x_off": x_off,
                "y_off": y_off,
                "tile_size": TILE_SIZE,
                "valid_width": valid_width,
                "valid_height": valid_height,
                "is_edge": valid_width < TILE_SIZE or valid_height < TILE_SIZE,
                "crs": src.crs.to_string() if src.crs else None,
                "west": west,
                "south": south,
                "east": east,
                "north": north,
            })

index = pd.DataFrame(records)
index.to_csv(INDEX_PATH, index=False)

summary = {
    "input_path": str(INPUT_PATH),
    "output_root": str(OUTPUT_ROOT),
    "tile_size": TILE_SIZE,
    "source_width": width,
    "source_height": height,
    "n_cols": n_cols,
    "n_rows": n_rows,
    "n_tiles": int(len(index)),
    "edge_tiles": int(index["is_edge"].sum()),
    "crs": crs,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Saved {len(index)} PNG tiles to {IMAGE_DIR}")
print(f"Saved tile index: {INDEX_PATH}")
print(f"Saved summary: {SUMMARY_PATH}")
print(f"Edge tiles: {summary['edge_tiles']}")
index.head()

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.2))
axes = np.atleast_2d(axes)

for _, rec in index.iterrows():
    ax = axes[int(rec["row"]), int(rec["col"])]
    img = Image.open(IMAGE_DIR / rec["file_name"])
    ax.imshow(img)
    ax.set_title(rec["tile_id"], fontsize=9)
    ax.axis("off")

plt.tight_layout()
fig.savefig(CONTACT_SHEET_PATH, dpi=150)
plt.show()

print(f"Saved contact sheet: {CONTACT_SHEET_PATH}")

In [ ]:
edge_tiles = index[index["is_edge"]]
print(f"Total tiles: {len(index)}")
print(f"Full tiles: {len(index) - len(edge_tiles)}")
print(f"Edge tiles: {len(edge_tiles)}")
edge_tiles[["tile_id", "valid_width", "valid_height", "west", "south", "east", "north"]]